In [16]:
import pandas as pd
import numpy as np
import plotly.express as px

In [17]:
df = pd.read_csv("dataset.csv")
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [19]:
print("Quantidade de linhas e colunas:", df.shape)
print("\nColunas do dataset:")
print(df.columns.tolist())

Quantidade de linhas e colunas: (114000, 20)

Colunas do dataset:
['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


# 1. Quais são os 10 gêneros musicais com a maior média de popularidade?

In [28]:
top_generos = (
    df.groupby("track_genre", as_index=False)["popularity"]
    .mean()
    .sort_values("popularity", ascending=False)
    .head(10)
    .sort_values("popularity")
)

fig = px.bar(
    top_generos,
    x="popularity",
    y="track_genre",
    orientation="h",
    text=top_generos["popularity"].round(2),
    title="Top 10 gêneros musicais com maior média de popularidade",
    labels={
        "popularity": "Popularidade média",
        "track_genre": "Gênero musical"
    },
    color="popularity",
    color_continuous_scale="Greens"
)

fig.update_traces(
    hovertemplate="<b>Gênero:</b> %{y}<br><b>Popularidade média:</b> %{x:.2f}<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    height=550,
    showlegend=False
)

fig.show()


# 2. Qual é a distribuição do ritmo das músicas na plataforma? (BPM)

In [29]:
fig = px.histogram(
    df,
    x="tempo",
    nbins=45,
    marginal="box",
    title="Distribuição do ritmo das músicas em BPM",
    labels={
        "tempo": "BPM / Tempo",
        "count": "Quantidade de músicas"
    },
    color_discrete_sequence=["#276EF1"]
)

fig.update_traces(
    hovertemplate="<b>BPM:</b> %{x}<br><b>Quantidade:</b> %{y}<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    height=550,
    bargap=0.04
)

fig.show()

# 3. O efeito “show ao vivo”: a experiência live destrói o desempenho no streaming?

In [30]:
df["faixa_liveness"] = pd.cut(
    df["liveness"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=["Muito baixo", "Baixo", "Médio", "Alto", "Muito alto"],
    include_lowest=True
)

fig = px.violin(
    df,
    x="faixa_liveness",
    y="popularity",
    box=True,
    points=False,
    title="Distribuição da popularidade por nível de liveness",
    labels={
        "faixa_liveness": "Nível de característica ao vivo",
        "popularity": "Popularidade"
    },
    color="faixa_liveness"
)

fig.update_traces(
    hovertemplate="<b>Liveness:</b> %{x}<br><b>Popularidade:</b> %{y}<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    height=550,
    showlegend=False
)

fig.show()

# 4. Há relação direta entre "loudness" (barulho) e energy?

In [31]:
amostra = df.sample(5000, random_state=42)

fig = px.scatter(
    amostra,
    x="loudness",
    y="energy",
    color="popularity",
    trendline="ols",
    hover_data=["track_name", "artists", "track_genre", "popularity"],
    title="Relação entre loudness e energy",
    labels={
        "loudness": "Loudness",
        "energy": "Energy",
        "popularity": "Popularidade"
    },
    color_continuous_scale="Viridis"
)

fig.update_traces(
    marker=dict(size=6, opacity=0.45),
    hovertemplate=
    "<b>Música:</b> %{customdata[0]}<br>" +
    "<b>Artista:</b> %{customdata[1]}<br>" +
    "<b>Gênero:</b> %{customdata[2]}<br>" +
    "<b>Popularidade:</b> %{customdata[3]}<br>" +
    "<b>Loudness:</b> %{x:.2f}<br>" +
    "<b>Energy:</b> %{y:.3f}<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    height=600
)

fig.show()

# 5. Como se comporta a dançabilidade entre músicas explícitas e não explícitas?

In [32]:
df["explicit_label"] = df["explicit"].map({
    True: "Explícita",
    False: "Não explícita"
})

fig = px.box(
    df,
    x="explicit_label",
    y="danceability",
    color="explicit_label",
    points="outliers",
    title="Dançabilidade entre músicas explícitas e não explícitas",
    labels={
        "explicit_label": "Tipo de conteúdo",
        "danceability": "Danceability"
    },
    color_discrete_map={
        "Explícita": "#E95B4A",
        "Não explícita": "#276EF1"
    }
)

fig.update_traces(
    hovertemplate="<b>Tipo:</b> %{x}<br><b>Danceability:</b> %{y:.3f}<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    height=550,
    showlegend=False
)

fig.show()

# 6. Músicas mais energéticas são mais populares?

In [34]:
df["faixa_energy"] = pd.cut(
    df["energy"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=["Muito baixa", "Baixa", "Média", "Alta", "Muito alta"],
    include_lowest=True
)

popularidade_energy = (
    df.groupby("faixa_energy", as_index=False)["popularity"]
    .mean()
)

fig = px.bar(
    popularidade_energy,
    x="faixa_energy",
    y="popularity",
    text=popularidade_energy["popularity"].round(2),
    title="Popularidade média por nível de energia",
    labels={
        "faixa_energy": "Nível de energia",
        "popularity": "Popularidade média"
    },
    color="popularity",
    color_continuous_scale="Oranges"
)

fig.update_traces(
    hovertemplate="<b>Energia:</b> %{x}<br><b>Popularidade média:</b> %{y:.2f}<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    height=500
)

fig.show()

/tmp/ipykernel_6919/1076692592.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



# 7. O "Efeito TikTok": Músicas mais curtas são mais populares na era do streaming?

In [35]:
df["duration_min"] = df["duration_ms"] / 60000

df["faixa_duracao"] = pd.cut(
    df["duration_min"],
    bins=[0, 2, 3, 4, 5, df["duration_min"].max()],
    labels=["Até 2 min", "2 a 3 min", "3 a 4 min", "4 a 5 min", "Mais de 5 min"],
    include_lowest=True
)

fig = px.box(
    df,
    x="faixa_duracao",
    y="popularity",
    color="faixa_duracao",
    points=False,
    title="Popularidade por faixa de duração da música",
    labels={
        "faixa_duracao": "Duração da música",
        "popularity": "Popularidade"
    }
)

fig.update_traces(
    hovertemplate="<b>Duração:</b> %{x}<br><b>Popularidade:</b> %{y}<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    height=550,
    showlegend=False
)

fig.show()

# 8. O Paradoxo da Acústica: O que acontece com a Energia quando os instrumentos são acústicos?

In [37]:
df["faixa_acousticness"] = pd.cut(
    df["acousticness"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=["Muito baixa", "Baixa", "Média", "Alta", "Muito alta"],
    include_lowest=True
)

energia_acustica = (
    df.groupby("faixa_acousticness", as_index=False)["energy"]
    .mean()
)

fig = px.line(
    energia_acustica,
    x="faixa_acousticness",
    y="energy",
    markers=True,
    title="Energia média por nível de acousticness",
    labels={
        "faixa_acousticness": "Nível de acousticness",
        "energy": "Energia média"
    }
)

fig.update_traces(
    line=dict(width=4, color="#C88A12"),
    marker=dict(size=10),
    hovertemplate="<b>Acousticness:</b> %{x}<br><b>Energia média:</b> %{y:.3f}<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    height=500
)

fig.show()

/tmp/ipykernel_6919/2974707615.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

